In [1]:
!pip install pandas scikit-learn nltk matplotlib seaborn

In [2]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
import seaborn as sns

import nltk
import re

In [3]:
df = pd.read_csv("upcoming.csv")

df.head()

,R_fighter,B_fighter,R_odds,B_odds,R_ev,B_ev,date,location,country,Winner,...,finish_details,finish_round,finish_round_time,total_fight_time_secs,r_dec_odds,b_dec_odds,r_sub_odds,b_sub_odds,r_ko_odds,b_ko_odds
0,Renato Moicano,Chris Duncan,150,-188,150.0000,53.1915,2026-04-04,"Las Vegas, Nevada, USA",USA,NaN,...,NaN,NaN,NaN,NaN,650.0,333.0,350.0,750.0,800.0,125.0
1,Virna Jandiroba,Tabatha Ricci,-120,-105,83.3333,95.2381,2026-04-04,"Las Vegas, Nevada, USA",USA,NaN,...,NaN,NaN,NaN,NaN,150.0,120.0,400.0,1800.0,1600.0,1400.0
2,Abdul Rakhman Yakhyaev,Brendson Ribeiro,-2000,900,5.0000,900.0000,2026-04-04,"Las Vegas, Nevada, USA",USA,NaN,...,NaN,NaN,NaN,NaN,750.0,1800.0,138.0,2500.0,-133.0,1800.0
3,Ethyn Ewing,Rafael Estevam,-150,120,66.6667,120.0000,2026-04-04,"Las Vegas, Nevada, USA",USA,NaN,...,NaN,NaN,NaN,NaN,163.0,225.0,1600.0,650.0,275.0,1000.0
4,Tommy McMillen,Manolo Zecchini,-1408,750,7.1023,750.0000,2026-04-04,"Las Vegas, Nevada, USA",USA,NaN,...,NaN,NaN,NaN,NaN,500.0,1400.0,-150.0,2800.0,200.0,1400.0


In [5]:
df.columns

Index(['R_fighter', 'B_fighter', 'R_odds', 'B_odds', 'R_ev', 'B_ev', 'date',
       'location', 'country', 'Winner',
       ...
       'finish_details', 'finish_round', 'finish_round_time',
       'total_fight_time_secs', 'r_dec_odds', 'b_dec_odds', 'r_sub_odds',
       'b_sub_odds', 'r_ko_odds', 'b_ko_odds'],
      dtype='str', length=118)

In [6]:
text_cols = df.select_dtypes(include=["object", "string"]).columns
print(text_cols)

Index(['R_fighter', 'B_fighter', 'date', 'location', 'country', 'weight_class',
       'gender', 'B_Stance', 'R_Stance'],
      dtype='str')


In [9]:
print(df.columns.tolist())
print(df.head())

['R_fighter', 'B_fighter', 'R_odds', 'B_odds', 'R_ev', 'B_ev', 'date', 'location', 'country', 'Winner', 'title_bout', 'weight_class', 'gender', 'no_of_rounds', 'B_current_lose_streak', 'B_current_win_streak', 'B_draw', 'B_avg_SIG_STR_landed', 'B_avg_SIG_STR_pct', 'B_avg_SUB_ATT', 'B_avg_TD_landed', 'B_avg_TD_pct', 'B_longest_win_streak', 'B_losses', 'B_total_rounds_fought', 'B_total_title_bouts', 'B_win_by_Decision_Majority', 'B_win_by_Decision_Split', 'B_win_by_Decision_Unanimous', 'B_win_by_KO/TKO', 'B_win_by_Submission', 'B_win_by_TKO_Doctor_Stoppage', 'B_wins', 'B_Stance', 'B_Height_cms', 'B_Reach_cms', 'B_Weight_lbs', 'R_current_lose_streak', 'R_current_win_streak', 'R_draw', 'R_avg_SIG_STR_landed', 'R_avg_SIG_STR_pct', 'R_avg_SUB_ATT', 'R_avg_TD_landed', 'R_avg_TD_pct', 'R_longest_win_streak', 'R_losses', 'R_total_rounds_fought', 'R_total_title_bouts', 'R_win_by_Decision_Majority', 'R_win_by_Decision_Split', 'R_win_by_Decision_Unanimous', 'R_win_by_KO/TKO', 'R_win_by_Submission',

In [11]:
print(df.columns.tolist())
print(df.head(3))

['R_fighter', 'B_fighter', 'R_odds', 'B_odds', 'R_ev', 'B_ev', 'date', 'location', 'country', 'Winner', 'title_bout', 'weight_class', 'gender', 'no_of_rounds', 'B_current_lose_streak', 'B_current_win_streak', 'B_draw', 'B_avg_SIG_STR_landed', 'B_avg_SIG_STR_pct', 'B_avg_SUB_ATT', 'B_avg_TD_landed', 'B_avg_TD_pct', 'B_longest_win_streak', 'B_losses', 'B_total_rounds_fought', 'B_total_title_bouts', 'B_win_by_Decision_Majority', 'B_win_by_Decision_Split', 'B_win_by_Decision_Unanimous', 'B_win_by_KO/TKO', 'B_win_by_Submission', 'B_win_by_TKO_Doctor_Stoppage', 'B_wins', 'B_Stance', 'B_Height_cms', 'B_Reach_cms', 'B_Weight_lbs', 'R_current_lose_streak', 'R_current_win_streak', 'R_draw', 'R_avg_SIG_STR_landed', 'R_avg_SIG_STR_pct', 'R_avg_SUB_ATT', 'R_avg_TD_landed', 'R_avg_TD_pct', 'R_longest_win_streak', 'R_losses', 'R_total_rounds_fought', 'R_total_title_bouts', 'R_win_by_Decision_Majority', 'R_win_by_Decision_Split', 'R_win_by_Decision_Unanimous', 'R_win_by_KO/TKO', 'R_win_by_Submission',

In [13]:
df.columns

Index(['R_fighter', 'B_fighter', 'R_odds', 'B_odds', 'R_ev', 'B_ev', 'date',
       'location', 'country', 'Winner',
       ...
       'finish_details', 'finish_round', 'finish_round_time',
       'total_fight_time_secs', 'r_dec_odds', 'b_dec_odds', 'r_sub_odds',
       'b_sub_odds', 'r_ko_odds', 'b_ko_odds'],
      dtype='str', length=118)

In [14]:
df.to_csv("clustered_upcoming.csv", index=False)